## RQ2

Für **jede Großstadt** wird gesucht nach:

1. der nächstgelegenen **Mittelstadt** mit 20.000–149.999 Einwohnern,
2. der nächstgelegenen **Kleinstadt** mit 5.000–19.999 Einwohnern,
3. der nächstgelegenen **eigenständigen Gemeinde** mit 1–4.999 Einwohnern.



In [1]:
import requests
import json
import time
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)

## 1. Alle deutschen Gemeinden auf einmal laden 
### Note 
1. Die Idee ist, dass alle deutsche Gemeinde zuerst laden und local die informationen haben, danach können wir es local bearbeiten und filtern, anstatt jedes mal eine Anfrage an DB zu schiecken, weil die Ausführung sehr lange dauert.
2. Die Anfrage, damit wir alle deutsche gemeinde auf einmal laden können ist sehr groß für DB und oft bekommen wir die Fehler mit HTTPError,   daher laden wir sie adaptive. 

In [ ]:
WIKIDATA_URL = "https://query.wikidata.org/sparql"

HEADERS = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "student-climate-project/1.0"
}


def _fetch_population_range(min_pop, max_pop):
    query = f"""
    SELECT DISTINCT ?place ?placeLabel ?population ?coord ?municipalityKey ?isCity WHERE {{
      ?place wdt:P17 wd:Q183;                     # liegt in Deutschland
             wdt:P31/wdt:P279* wd:Q262166;         # eigenständige deutsche Gemeinde
             wdt:P1082 ?population;                # hat Einwohnerzahl
             wdt:P625 ?coord;                      # hat Koordinaten
             wdt:P439 ?municipalityKey.             # offizieller Gemeindeschlüssel

      BIND(EXISTS {{ ?place wdt:P31/wdt:P279* wd:Q515. }} AS ?isCity)

      FILTER(?population >= {min_pop} && ?population <= {max_pop})

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "de,en". }}
    }}
    """

    r = requests.get(WIKIDATA_URL, params={"query": query}, headers=HEADERS, timeout=180)
    r.raise_for_status()

    payload = json.loads(r.text, strict=False)
    return payload["results"]["bindings"]


def _fetch_population_range_adaptive(min_pop, max_pop, depth=0, max_depth=10):
    try:
        rows = _fetch_population_range(min_pop, max_pop)
        print(f"  ... Population {min_pop}-{max_pop}: {len(rows)} Gemeinden")
        return rows
    except (requests.HTTPError, requests.exceptions.Timeout, json.JSONDecodeError) as e:
        if depth >= max_depth or max_pop <= min_pop:
            raise RuntimeError(
                f"Populationsbereich {min_pop}-{max_pop} lässt sich auch nach {depth} "
                f"Splits nicht laden. Original-Fehler: {e}"
            ) from e

        mid = (min_pop + max_pop) // 2
        print(f"  ... Population {min_pop}-{max_pop} fehlgeschlagen ({e}), splitte bei {mid}")
        time.sleep(1)

        left = _fetch_population_range_adaptive(min_pop, mid, depth + 1, max_depth)
        right = _fetch_population_range_adaptive(mid + 1, max_pop, depth + 1, max_depth)
        return left + right


def get_all_german_municipalities(min_population=1, max_population=4_000_000):

    rows = _fetch_population_range_adaptive(min_population, max_population)

    places = []
    for row in rows:
        name = row["placeLabel"]["value"]
        pop = int(float(row["population"]["value"]))
        lon, lat = row["coord"]["value"].replace("Point(", "").replace(")", "").split(" ")
        places.append({
            "place": name,
            "wikidata_id": row["place"]["value"].split("/")[-1],
            "population": pop,
            "lat": float(lat),
            "lon": float(lon),
            "municipality_key": row["municipalityKey"]["value"],
            "is_city": row["isCity"]["value"] == "true"
        })

    # Gemeindeschlüssel ist eindeutiger als nur der Name
    df = pd.DataFrame(places).drop_duplicates(subset="municipality_key")
    return df.reset_index(drop=True)


all_places_df = get_all_german_municipalities(min_population=1)

print(f"Insgesamt geladen: {len(all_places_df)} eigenständige deutsche Gemeinden")
all_places_df.head()

  ... Population 1-4000000 fehlgeschlagen (Expecting ',' delimiter: line 289574 column 1 (char 8018246)), splitte bei 2000000
  ... Population 1-2000000: 11345 Gemeinden
  ... Population 2000001-4000000: 1 Gemeinden
Insgesamt geladen: 11119 eigenständige deutsche Gemeinden


,place,wikidata_id,population,lat,lon,municipality_key,is_city
0,Brühl,Q7036,45515,50.833333,6.900000,05362012,True
1,Vilsbiburg,Q521018,12621,48.447450,12.347500,09274184,True
2,Schwarzatal,Q59775088,3355,50.583333,11.150000,16073113,True
3,Ichenhausen,Q503202,9403,48.371190,10.307060,09774143,True
4,Waldmohr,Q552671,5281,49.395278,7.341111,07336102,True


## 2. Großstädte aus dem Datensatz herausfiltern
Hier filtern wir alle deutsche Städte mit Einwohneranzahl >= 150000 

In [3]:
cities_df = (
    all_places_df[all_places_df["population"] >= 150000]
    .sort_values("population", ascending=False)
    .reset_index(drop=True)
)

print(cities_df)
print(f"Anzahl Großstädte: {len(cities_df)}")

                    place wikidata_id  population        lat        lon  \
0                  Berlin         Q64     3782202  52.516667  13.383333   
1                 Hamburg       Q1055     1910160  53.550000  10.000000   
2                 München       Q1726     1510378  48.137500  11.575000   
3                    Köln        Q365     1024621  50.942222   6.957778   
4       Frankfurt am Main       Q1794      775790  50.110556   8.682222   
5               Stuttgart       Q1022      633484  48.777500   9.180000   
6              Düsseldorf       Q1718      631217  51.225556   6.776667   
7                 Leipzig       Q2079      611850  51.340632  12.374733   
8                Dortmund       Q1295      595471  51.513889   7.465278   
9                   Essen       Q2066      586608  51.450833   7.013056   
10                 Bremen      Q24879      577026  53.075833   8.807222   
11                Dresden       Q1731      564904  51.049329  13.738144   
12               Hannover

## 3. Funktion: Orte in der Nähe einer Großstadt suchen (jetzt lokal)

`find_places_near_city` nutzt die `haversine_km` Funktion, die den Abstand in Km zwischen zwei punkte berechnet und gibt alle Orte, die in der Nähe einer Großstadt sind. 

In [4]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def find_places_near_city(
    lat,
    lon,
    min_pop,
    max_pop=None,
    radius_km=50,
    place_kind="city"
):
    
    if place_kind == "city":
        df = all_places_df[all_places_df["is_city"]]
    elif place_kind == "rural":
        df = all_places_df[~all_places_df["is_city"]]
    else:
        raise ValueError("place_kind must be 'city' or 'rural'")

    df = df[df["population"] >= min_pop]
    if max_pop is not None:
        df = df[df["population"] <= max_pop]

    if df.empty:
        return df

    distances = haversine_km(lat, lon, df["lat"].to_numpy(), df["lon"].to_numpy()) 

    result = df.copy()
    result["distance_km"] = distances
    result = result[result["distance_km"] <= radius_km]

    if result.empty:
        return result

    return (
        result.drop_duplicates(subset="municipality_key")
              .sort_values("distance_km")
              .reset_index(drop=True)
    )

## 4. Ermittlung des nächstgelegenen Vergleichsortes
Die Funktion sucht den nächstgelegenen passenden Vergleichsort. Falls kein geeigneter Ort gefunden wird, wird der Suchradius schrittweise erweitert und wenn keinen Ort nach den Radius `radii = [30, 50, 75, 100, 150, 200]` gefunden wurde, dann gibt None zurück. 

In [5]:
def get_nearest_comparison_place(
    lat,
    lon,
    min_pop,
    max_pop,
    large_city_name,
    place_kind,
    min_distance_km=0
):
    radii = [30, 50, 75, 100, 150, 200]

    for radius in radii:
        df = find_places_near_city(
            lat=lat,
            lon=lon,
            min_pop=min_pop,
            max_pop=max_pop,
            radius_km=radius,
            place_kind=place_kind
        )

        if not df.empty:
            df = df[
                (df["place"].str.lower() != large_city_name.lower()) &
                (df["distance_km"] >= min_distance_km)
            ].copy()

            if not df.empty:
                result = df.iloc[0].copy()
                result["search_radius_km"] = radius
                return result

    return None

## 5. Für jede Großstadt Medium, Small und Rural bestimmen

Für jede Großstadt wird eine Mittelstadt, Kleinstadt und Gemeinde gefunden. 

In [6]:
matches = []

RURAL_MIN_DISTANCE_KM = 15

for _, row in cities_df.iterrows():
    large_city = row["place"]
    large_pop = int(row["population"])
    lat = row["lat"]
    lon = row["lon"]

    print(f"Suche Vergleichsorte für {large_city} ...")

    categories = {
        "medium": {
            "min_pop": 20000,
            "max_pop": 149999,
            "place_kind": "city",
            "min_distance_km": 0
        },
        "small": {
            "min_pop": 5000,
            "max_pop": 19999,
            "place_kind": "city",
            "min_distance_km": 0
        },
        "rural": {
            "min_pop": 1,
            "max_pop": 4999,
            "place_kind": "rural",
            "min_distance_km": RURAL_MIN_DISTANCE_KM
        }
    }

    matches.append({
        "large_city": large_city,
        "large_city_population": large_pop,
        "comparison_place": large_city,
        "size_class": "large",
        "population": large_pop,
        "distance_km": 0.0,
        "search_radius_km": 0,
        "lat": lat,
        "lon": lon,
        "municipality_key": row.get("municipality_key", None),
        "wikidata_id": row.get("wikidata_id", None)
    })

    for size_class, settings in categories.items():
        result = get_nearest_comparison_place(
            lat=lat,
            lon=lon,
            min_pop=settings["min_pop"],
            max_pop=settings["max_pop"],
            large_city_name=large_city,
            place_kind=settings["place_kind"],
            min_distance_km=settings["min_distance_km"]
        )

        if result is not None:
            matches.append({
                "large_city": large_city,
                "large_city_population": large_pop,
                "comparison_place": result["place"],
                "size_class": size_class,
                "population": int(result["population"]),
                "distance_km": float(result["distance_km"]),
                "search_radius_km": int(result["search_radius_km"]),
                "lat": float(result["lat"]),
                "lon": float(result["lon"]),
                "municipality_key": result["municipality_key"],
                "wikidata_id": result["wikidata_id"]
            })
        else:
            print(f"  Kein gültiger {size_class}-Treffer für {large_city} gefunden.")

comparison_df = pd.DataFrame(matches)
comparison_df

Suche Vergleichsorte für Berlin ...
Suche Vergleichsorte für Hamburg ...
Suche Vergleichsorte für München ...
Suche Vergleichsorte für Köln ...
Suche Vergleichsorte für Frankfurt am Main ...
Suche Vergleichsorte für Stuttgart ...
Suche Vergleichsorte für Düsseldorf ...
Suche Vergleichsorte für Leipzig ...
Suche Vergleichsorte für Dortmund ...
Suche Vergleichsorte für Essen ...
Suche Vergleichsorte für Bremen ...
Suche Vergleichsorte für Dresden ...
Suche Vergleichsorte für Hannover ...
Suche Vergleichsorte für Nürnberg ...
Suche Vergleichsorte für Duisburg ...
Suche Vergleichsorte für Bochum ...
Suche Vergleichsorte für Wuppertal ...
Suche Vergleichsorte für Bielefeld ...
Suche Vergleichsorte für Bonn ...
Suche Vergleichsorte für Münster ...
Suche Vergleichsorte für Mannheim ...
Suche Vergleichsorte für Karlsruhe ...
Suche Vergleichsorte für Augsburg ...
Suche Vergleichsorte für Wiesbaden ...
Suche Vergleichsorte für Mönchengladbach ...
Suche Vergleichsorte für Gelsenkirchen ...
Suche 

,large_city,large_city_population,comparison_place,size_class,population,distance_km,search_radius_km,lat,lon,municipality_key,wikidata_id
0,Berlin,3782202,Berlin,large,3782202,0.000000,0,52.516667,13.383333,11000000,Q64
1,Berlin,3782202,Teltow,medium,27880,14.843479,30,52.402222,13.270556,12069616,Q572512
2,Berlin,3782202,Velten,small,12733,23.976321,30,52.691389,13.175278,12065332,Q585613
3,Berlin,3782202,Gosen-Neu Zittau,rural,3428,26.199325,30,52.393333,13.712778,12067173,Q624117
4,Hamburg,1910160,Hamburg,large,1910160,0.000000,0,53.550000,10.000000,02000000,Q1055
...,...,...,...,...,...,...,...,...,...,...,...
219,Neuss,155163,Berg,rural,1306,73.855658,75,50.555556,6.946944,07131011,Q647551
220,Regensburg,151517,Regensburg,large,151517,0.000000,0,49.016667,12.083333,09362000,Q2978
221,Regensburg,151517,Schwandorf,medium,30239,34.149335,50,49.323600,12.099350,09376161,Q504768
222,Regensburg,151517,Neutraubling,small,14614,8.850619,30,48.987778,12.196389,09375174,Q488625


## 6. Übersichtliche Matching-Tabelle

In [7]:
# Großstädten nach Einwohneranzahl sortieren 
city_order = (
    cities_df
    .sort_values("population", ascending=False)["place"]
    .tolist()
)

# Nur die drei Vergleichsgruppen anzeigen
comparison_table = comparison_df[
    comparison_df["size_class"].isin(["medium", "small", "rural"])
].pivot(
    index="large_city",
    columns="size_class",
    values="comparison_place"
)

# Die spalten werden nach medium, small und rural sortiert 
comparison_table = comparison_table[
    ["medium", "small", "rural"]
]

# Großstädte werden nach Einwohnerzahl sortiert 
comparison_table = comparison_table.reindex(city_order)

# Deutsche Name für die Spalten 
comparison_table = comparison_table.rename(
    columns={
        "medium": "Mittelstadt",
        "small": "Kleinstadt",
        "rural": "Gemeinde"
    }
)

# Index benennen
comparison_table.index.name = "Großstadt"

comparison_table

size_class,Mittelstadt,Kleinstadt,Gemeinde
Großstadt,,,
Berlin,Teltow,Velten,Gosen-Neu Zittau
Hamburg,Reinbek,Schenefeld,Stapelfeld
München,Haar,Garching bei München,Baierbrunn
Köln,Hürth,Burscheid,Rheinbreitbach
Frankfurt am Main,Offenbach am Main,Steinbach (Taunus),Messel
Stuttgart,Korntal-Münchingen,Gerlingen,Affalterbach
Düsseldorf,Meerbusch,Burscheid,Buchholz
Leipzig,Markkleeberg,Taucha,Jesewitz
Dortmund,Schwerte,Olfen,Friesenhagen


## 7. Ergebnis als CSV speichern

In [ ]:
comparison_df.to_csv(
    "RQ2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Gespeichert: RQ2.csv")

Gespeichert: RQ2.csv
